# Notebook 5 — News Aggregator

Pulls latest news headlines for:
- Each sector in your anomaly windows
- Each candidate stock
- Macro: RBI, government policy, budget, FII flows

**Sources (all free, no API key needed):**
- Google News RSS
- RBI official RSS
- BSE announcements (via Google News)
- yfinance `.news`

**Run this:** ~1 week before a window opens, and again on entry day.

In [ ]:
# !pip install yfinance pandas requests feedparser openpyxl

In [1]:
import feedparser
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from urllib.parse import quote
import time, os, warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


In [2]:
# ── Smart Configuration — auto-detects upcoming windows from anomalies.xlsx ──
#
# Instead of manually setting ACTIVE_WINDOW, this cell:
#   1. Reads your anomalies.xlsx (output from Notebook 3)
#   2. Filters by score threshold
#   3. Finds all windows starting within LOOKAHEAD_DAYS from today
#   4. Deduplicates overlapping windows (same sector, same week)
#   5. Auto-selects the next upcoming window
#   6. Builds the WINDOWS + ACTIVE_WINDOW config that the rest of the notebook uses

import pandas as pd
from datetime import datetime

# ── Parameters — adjust these ─────────────────────────────────────────────────
ANOMALIES_FILE     = 'output/anomalies.xlsx'  # path to your Notebook 3 output
SCORE_THRESHOLD    = 0.30   # minimum composite score to consider
LOOKAHEAD_DAYS     = 30     # how many days ahead to scan for upcoming windows
GROUPING_DAYS      = 7      # windows within this many days = same cluster (deduped)
NEWS_LOOKBACK_DAYS = 7      # how many days back to pull news
OUTPUT_DIR         = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Sector metadata: search queries + stock tickers ───────────────────────────
# Used to build the WINDOWS config automatically once a sector is detected
SECTOR_META = {
    'Nifty FMCG': {
        'sector_queries': ['Nifty FMCG', 'FMCG India sector', 'India consumer staples'],
        'stocks': {
            'HINDUNILVR.NS': 'Hindustan Unilever HUL',
            'ITC.NS'       : 'ITC Limited India',
            'NESTLEIND.NS' : 'Nestle India',
            'BRITANNIA.NS' : 'Britannia Industries',
            'DABUR.NS'     : 'Dabur India',
            'MARICO.NS'    : 'Marico India',
            'GODREJCP.NS'  : 'Godrej Consumer Products',
            'COLPAL.NS'    : 'Colgate Palmolive India',
            'TATACONSUM.NS': 'Tata Consumer Products',
        },
    },
    'Nifty Pharma': {
        'sector_queries': ['Nifty Pharma India', 'India pharma sector', 'US FDA India drugs'],
        'stocks': {
            'SUNPHARMA.NS' : 'Sun Pharma India',
            'DRREDDY.NS'   : "Dr Reddy's Laboratories",
            'CIPLA.NS'     : 'Cipla India',
            'DIVISLAB.NS'  : "Divi's Laboratories",
            'AUROPHARMA.NS': 'Aurobindo Pharma',
            'LUPIN.NS'     : 'Lupin Limited India',
            'TORNTPHARM.NS': 'Torrent Pharma',
        },
    },
    'Nifty Energy': {
        'sector_queries': ['Nifty Energy India', 'India oil gas sector', 'crude oil India'],
        'stocks': {
            'RELIANCE.NS': 'Reliance Industries',
            'ONGC.NS'    : 'ONGC India',
            'NTPC.NS'    : 'NTPC India power',
            'BPCL.NS'    : 'BPCL India',
            'IOC.NS'     : 'Indian Oil Corporation',
            'GAIL.NS'    : 'GAIL India gas',
        },
    },
    'Nifty Bank': {
        'sector_queries': ['Nifty Bank India', 'India banking sector', 'RBI banking policy'],
        'stocks': {
            'HDFCBANK.NS' : 'HDFC Bank',
            'ICICIBANK.NS': 'ICICI Bank',
            'SBIN.NS'     : 'State Bank of India',
            'KOTAKBANK.NS': 'Kotak Mahindra Bank',
            'AXISBANK.NS' : 'Axis Bank',
            'INDUSINDBK.NS':'IndusInd Bank',
        },
    },
    'Nifty Auto': {
        'sector_queries': ['Nifty Auto India', 'India automobile sector', 'India auto sales'],
        'stocks': {
            'MARUTI.NS'   : 'Maruti Suzuki',
            'TATAMOTORS.NS': 'Tata Motors',
            'M&M.NS'      : 'Mahindra and Mahindra',
            'BAJAJ-AUTO.NS': 'Bajaj Auto',
            'HEROMOTOCO.NS': 'Hero MotoCorp',
            'EICHERMOT.NS': 'Eicher Motors',
        },
    },
    'Nifty Realty': {
        'sector_queries': ['Nifty Realty India', 'India real estate sector', 'India property market'],
        'stocks': {
            'DLF.NS'         : 'DLF Limited',
            'GODREJPROP.NS'  : 'Godrej Properties',
            'OBEROIRLTY.NS'  : 'Oberoi Realty',
            'PRESTIGE.NS'    : 'Prestige Estates',
            'PHOENIXLTD.NS'  : 'Phoenix Mills',
        },
    },
    'Nifty Financial Services': {
        'sector_queries': ['Nifty Financial Services India', 'India NBFC sector', 'India fintech'],
        'stocks': {
            'BAJFINANCE.NS' : 'Bajaj Finance',
            'BAJAJFINSV.NS' : 'Bajaj Finserv',
            'HDFCLIFE.NS'   : 'HDFC Life Insurance',
            'SBILIFE.NS'    : 'SBI Life Insurance',
            'ICICIGI.NS'    : 'ICICI Lombard',
            'MUTHOOTFIN.NS' : 'Muthoot Finance',
        },
    },
    'Nifty Metal': {
        'sector_queries': ['Nifty Metal India', 'India steel sector', 'India metals commodities'],
        'stocks': {
            'TATASTEEL.NS'  : 'Tata Steel',
            'JSWSTEEL.NS'   : 'JSW Steel',
            'HINDALCO.NS'   : 'Hindalco Industries',
            'SAIL.NS'       : 'Steel Authority of India',
            'VEDL.NS'       : 'Vedanta Limited',
        },
    },
    'Nifty Media': {
        'sector_queries': ['Nifty Media India', 'India media entertainment sector'],
        'stocks': {
            'ZEEL.NS'    : 'Zee Entertainment',
            'SUNTV.NS'   : 'Sun TV Network',
            'PVRINOX.NS' : 'PVR Inox',
            'NAZARA.NS'  : 'Nazara Technologies',
        },
    },
}

MACRO_QUERIES = [
    'RBI monetary policy India',
    'India government budget policy',
    'India import export duty',
    'India inflation CPI WPI',
    'India FII FDI foreign investment',
    'India GST policy change',
]

# ── Load anomalies and detect upcoming windows ────────────────────────────────
today    = datetime.today()
year     = today.year

print(f'Today          : {today.strftime("%Y-%m-%d")}')
print(f'Score threshold: >= {SCORE_THRESHOLD}')
print(f'Lookahead      : {LOOKAHEAD_DAYS} days\n')

try:
    anomalies_df = pd.read_excel(ANOMALIES_FILE, sheet_name='All Anomalies')
except FileNotFoundError:
    raise FileNotFoundError(
        f'Could not find {ANOMALIES_FILE}. '
        'Make sure you have run Notebook 3 and the file is in your output/ folder.'
    )

# Filter by score
qualified = anomalies_df[anomalies_df['Score'] >= SCORE_THRESHOLD].copy()
print(f'Anomalies above score {SCORE_THRESHOLD}: {len(qualified):,}')

# Parse dates and find upcoming windows
upcoming_rows = []
for _, row in qualified.iterrows():
    try:
        start = pd.to_datetime(f"{row['Window Start']} {year}", format='%b %d %Y')
        # If already passed this year, look at next year
        if start.date() < today.date():
            start = pd.to_datetime(f"{row['Window Start']} {year + 1}", format='%b %d %Y')
        days_until = (start.date() - today.date()).days
        if 0 <= days_until <= LOOKAHEAD_DAYS:
            upcoming_rows.append({
                'sector'      : row['Sector'],
                'start_label' : row['Window Start'],
                'start_date'  : start,
                'window_days' : int(row['Window Days']),
                'days_until'  : days_until,
                'score'       : row['Score'],
                'win_rate'    : row['Win Rate %'],
                'avg_return'  : row['Avg Return %'],
                'sharpe'      : row['Sharpe'],
            })
    except Exception:
        continue

if not upcoming_rows:
    print(f'\n⚠ No windows with score >= {SCORE_THRESHOLD} found in the next {LOOKAHEAD_DAYS} days.')
    print('Try increasing LOOKAHEAD_DAYS or lowering SCORE_THRESHOLD.')
    raise SystemExit('Nothing to run news for right now.')

upcoming = pd.DataFrame(upcoming_rows).sort_values('score', ascending=False)

# Deduplicate: keep best-scoring window per (sector, calendar week)
upcoming['week'] = upcoming['start_date'].dt.to_period('W')
best_per_cluster = (
    upcoming
    .groupby(['sector', 'week'], sort=False)
    .first()
    .reset_index()
    .sort_values('days_until')
)

# ── Print upcoming windows table ──────────────────────────────────────────────
print(f'\n{"═"*72}')
print(f'  UPCOMING ANOMALY WINDOWS — next {LOOKAHEAD_DAYS} days')
print(f'{"═"*72}')
print(f'  {"#":<3} {"Sector":<28} {"Start":>7} {"In":>4} {"Days":>5} {"Score":>6} {"WinRate":>8} {"Return":>7}')
print(f'  {"─"*70}')
for i, row in best_per_cluster.iterrows():
    marker = ' ◄ AUTO-SELECTED' if i == best_per_cluster.index[0] else ''
    print(
        f'  {best_per_cluster.index.get_loc(i)+1:<3} '
        f'{row["sector"]:<28} '
        f'{row["start_label"]:>7} '
        f'{int(row["days_until"]):>3}d '
        f'{int(row["window_days"]):>5}d '
        f'{row["score"]:>6.3f} '
        f'{row["win_rate"]:>7.0f}% '
        f'{row["avg_return"]:>6.2f}%'
        f'{marker}'
    )
print(f'{"═"*72}')

# ── Auto-select the soonest upcoming window (or manually override below) ──────
# To override: set OVERRIDE_INDEX to the # from the table above (1-based)
OVERRIDE_INDEX = None   # e.g. set to 3 to pick the 3rd row instead of auto

if OVERRIDE_INDEX is not None:
    selected_row = best_per_cluster.iloc[OVERRIDE_INDEX - 1]
    print(f'\n⚙ Manual override: selected row #{OVERRIDE_INDEX}')
else:
    selected_row = best_per_cluster.iloc[0]
    print(f'\n✓ Auto-selected: soonest upcoming window')

sector_name   = selected_row['sector']
start_label   = selected_row['start_label']
window_days   = int(selected_row['window_days'])
days_until    = int(selected_row['days_until'])

# Build ACTIVE_WINDOW name and WINDOWS config dynamically
ACTIVE_WINDOW = f"{sector_name} — {start_label} ({window_days}d)"

if sector_name not in SECTOR_META:
    raise KeyError(
        f'Sector "{sector_name}" not in SECTOR_META. '
        f'Add its stock tickers and search queries to SECTOR_META above.'
    )

WINDOWS = {
    ACTIVE_WINDOW: {
        'sector'        : sector_name,
        'sector_queries': SECTOR_META[sector_name]['sector_queries'],
        'stocks'        : SECTOR_META[sector_name]['stocks'],
    }
}

window_config = WINDOWS[ACTIVE_WINDOW]

print(f'\n  Window name  : {ACTIVE_WINDOW}')
print(f'  Sector       : {sector_name}')
print(f'  Start date   : {start_label}  ({days_until} days from today)')
print(f'  Window length: {window_days} trading days')
print(f'  Score        : {selected_row["score"]:.3f}  |  Win rate: {selected_row["win_rate"]:.0f}%  |  Avg return: {selected_row["avg_return"]:.2f}%')
print(f'  Stocks       : {len(window_config["stocks"])}')


Today          : 2026-05-25
Score threshold: >= 0.3
Lookahead      : 30 days

Anomalies above score 0.3: 2,735

════════════════════════════════════════════════════════════════════════
  UPCOMING ANOMALY WINDOWS — next 30 days
════════════════════════════════════════════════════════════════════════
  #   Sector                         Start   In  Days  Score  WinRate  Return
  ──────────────────────────────────────────────────────────────────────
  1   Nifty Auto                    May 25   0d    10d  0.343      90%   3.76% ◄ AUTO-SELECTED
  2   Nifty FMCG                    May 25   0d    36d  0.332      80%   5.60%
  3   Nifty Media                   May 25   0d    12d  0.321      90%   4.49%
  4   Nifty Realty                  May 29   4d     3d  0.340      90%   3.39%
  5   Nifty FMCG                    Jun 07  13d    27d  0.384      90%   3.56%
  6   Nifty Pharma                  Jun 07  13d    28d  0.699     100%   5.21%
  7   Nifty Pharma                  Jun 12  18d    25d  0.5

In [3]:
# ── Helper functions ──────────────────────────────────────────────────────────

def fetch_google_news(query, max_results=8):
    url     = f'https://news.google.com/rss/search?q={quote(query)}&hl=en-IN&gl=IN&ceid=IN:en'
    cutoff  = datetime.now() - timedelta(days=NEWS_LOOKBACK_DAYS)
    try:
        feed     = feedparser.parse(url)
        articles = []
        for entry in feed.entries[:max_results * 2]:
            try:
                pub = datetime(*entry.published_parsed[:6])
            except Exception:
                pub = datetime.now()
            if pub < cutoff:
                continue
            articles.append({
                'title'    : entry.get('title', 'No title'),
                'source'   : entry.get('source', {}).get('title', 'Unknown'),
                'published': pub.strftime('%Y-%m-%d %H:%M'),
                'link'     : entry.get('link', ''),
            })
            if len(articles) >= max_results:
                break
        return articles
    except Exception as e:
        print(f'  Error fetching "{query}": {e}')
        return []


def fetch_rbi_rss():
    cutoff   = datetime.now() - timedelta(days=NEWS_LOOKBACK_DAYS)
    articles = []
    for url in ['https://www.rbi.org.in/scripts/rss.aspx']:
        try:
            feed = feedparser.parse(url)
            for entry in feed.entries[:10]:
                try:
                    pub = datetime(*entry.published_parsed[:6])
                except Exception:
                    pub = datetime.now()
                if pub < cutoff:
                    continue
                articles.append({
                    'title'    : entry.get('title', 'No title'),
                    'source'   : 'RBI Official',
                    'published': pub.strftime('%Y-%m-%d %H:%M'),
                    'link'     : entry.get('link', ''),
                })
        except Exception:
            pass
    return articles


def fetch_yfinance_news(ticker, max_results=5):
    cutoff   = datetime.now() - timedelta(days=NEWS_LOOKBACK_DAYS)
    articles = []
    try:
        raw_news = yf.Ticker(ticker).news
        for item in raw_news[:max_results * 2]:
            try:
                pub = datetime.fromtimestamp(item.get('providerPublishTime', 0))
            except Exception:
                pub = datetime.now()
            if pub < cutoff:
                continue
            content = item.get('content', {})
            articles.append({
                'title'    : content.get('title') or item.get('title', 'No title'),
                'source'   : content.get('provider', {}).get('displayName') or item.get('publisher', 'yfinance'),
                'published': pub.strftime('%Y-%m-%d %H:%M'),
                'link'     : content.get('canonicalUrl', {}).get('url') or item.get('link', ''),
            })
            if len(articles) >= max_results:
                break
    except Exception:
        pass
    return articles


def print_articles(articles, indent=2):
    pad = ' ' * indent
    if not articles:
        print(f'{pad}No recent articles found.')
        return
    for i, a in enumerate(articles, 1):
        print(f"{pad}{i}. [{a['published']}] {a['source']}")
        print(f"{pad}   {a['title']}")
        if a['link']:
            print(f"{pad}   {a['link']}")
        print()


print('Helper functions defined.')

Helper functions defined.


In [4]:
# ── SECTION 1: Macro News ─────────────────────────────────────────────────────
# Read this first. A macro headwind can override any sector anomaly.

print('=' * 65)
print('  SECTION 1: MACRO — RBI, Policy, Budget, FII')
print('=' * 65)

all_news = []

print('\n── RBI Official Announcements ──')
rbi_news = fetch_rbi_rss()
print_articles(rbi_news)
for a in rbi_news:
    all_news.append({'Section': 'Macro', 'Category': 'RBI Official', 'Query': 'RBI RSS', **a})

for query in MACRO_QUERIES:
    print(f'\n── {query} ──')
    articles = fetch_google_news(query, max_results=5)
    print_articles(articles)
    for a in articles:
        all_news.append({'Section': 'Macro', 'Category': 'Google News', 'Query': query, **a})
    time.sleep(0.5)

  SECTION 1: MACRO — RBI, Policy, Budget, FII

── RBI Official Announcements ──
  No recent articles found.

── RBI monetary policy India ──
  1. [2026-05-24 17:39] Business Standard
     RBI's MPC likely to maintain status quo in June: Business standard poll - Business Standard
     https://news.google.com/rss/articles/CBMi0wFBVV95cUxQd0NndmtmOGFvVjZSazhmdGpwOGFQYVVWVGhPc1Nsd0lyYk9sU3U2alFCQ0cyM1BHb0xfN0RXSkFWWXBncUxUWXZueUFObjlfQmd6SjFoOU1hWVRmVzFycDE3aG0tUGxSUUxYNWMxR3dIRXdsSFdGS2FsNkNqQmk3MHppZTVfbXBtZXFQZU9GTlRBMkI2SC15dDN6WFFaZkprNUZFZmVWZGtEQ1BUT1pqT0t4cGJHNElrNTFFMU1PdGswY0MwcEhoVGlQMzZZbVItT04w0gHYAUFVX3lxTFAtS3NuZGNaRXlmSnFRLVJIRGlMbUlmUGlUWmtiM0ZzOFEwejRlTTNuczdreHNBaTMtU2ZfWTU5eXdOX2RJS0pWckI1N1hnVDJyUlU2WF91V1BIQ1pjWFVJM1hFOHZVVXhTTjgwWW5IRlZhdWJjUF9rd0p3Zmotdm96eHRBOUI1ZzFYTDNVSWF2WjczeUN1R3BtVzlPVUQxQnNlSDloaWNJb1ZRTlpLQVIyb0tXUGVsdVBKYml4ZUNyTzVFM0VCTVFvOHZ4eFBTUF8teXhxbWF3aw?oc=5

  2. [2026-05-22 17:29] The Economic Times
     India in better position to manage retail

In [5]:
# ── SECTION 2: Sector News ────────────────────────────────────────────────────
# Demand trends, policy changes, import/export affecting the sector.

print('=' * 65)
print(f'  SECTION 2: SECTOR — {window_config["sector"]}')
print('=' * 65)

for query in window_config['sector_queries']:
    print(f'\n── {query} ──')
    articles = fetch_google_news(query, max_results=6)
    print_articles(articles)
    for a in articles:
        all_news.append({'Section': 'Sector', 'Category': window_config['sector'], 'Query': query, **a})
    time.sleep(0.5)

  SECTION 2: SECTOR — Nifty Auto

── Nifty Auto India ──
  No recent articles found.

── India automobile sector ──
  1. [2026-05-20 05:50] IBEF
     Automotive Glass Manufacturing in India: Growth Drivers and Future Outlook - IBEF
     https://news.google.com/rss/articles/CBMiogFBVV95cUxPSWRSYnJ3WDFzMlJnQ2Exa09Gb2MtMXdCNlowb2dWdkQ1MS1zc0paN1hVZnhVMG91b0FvbGZVU1R4dW1rUi0wd21qbVdnSGJhX3NfUGtqazFxU3o0YndsWjJPNmx3VTNHdGN5cnFuX3ItMlVvNXRiRi0tNm8zakxMdjRac1VGTlgtbjBSVHhmOURJNmtXOUMzZ1YyMXpMUUttcHc?oc=5

  2. [2026-05-21 10:29] The Economic Times
     Auto sector still growing but these are the risks every investor must watch in FY27: Deep Shah - The Economic Times
     https://news.google.com/rss/articles/CBMi-wFBVV95cUxOajNHRE44SnFqXzBZbkljRnFUWWMydDAtUGRDUjFwUzVzWUpuYWV2TTVWZkR0WnZhSGtVVUFnYkZnQ1A3cXhqWjdoQmV4ZTlKTk5HUWhQeVhXbzMxMWUxeXVtQzkxaHRuZmRuWVN2UnVlVXBYSC1YbVpfRWphdGhwRmozN0lTMkdHZkxDVzhaYzV0UzNrVXVxZWF4M3BUcXRKemtmVnI5aEpsV0FtSE9EcEYtU2p4b0paeWVaeW9VTmxrbHMxRGNKdE9aSHJDVTVkSTh4LU

In [6]:
# ── SECTION 3: Company News ───────────────────────────────────────────────────
# Three sources per stock: Google News, BSE announcements, yfinance.
#
# What to look for:
#   RED    : fraud, investigation, ban, price control, debt default, FDA warning
#   YELLOW : results upcoming (binary risk), merger rumor, management change
#   GREEN  : contract win, approval, export growth, results beat

print('=' * 65)
print('  SECTION 3: COMPANY NEWS')
print('=' * 65)

for ticker, company_name in window_config['stocks'].items():
    print(f'\n{"─" * 55}')
    print(f'  {company_name} ({ticker})')
    print(f'  {"─" * 53}')

    print('  Google News:')
    g_art = fetch_google_news(company_name, max_results=5)
    print_articles(g_art, indent=4)
    for a in g_art:
        all_news.append({'Section': 'Company', 'Category': company_name, 'Query': 'Google News', **a})
    time.sleep(0.4)

    print('  BSE Announcements:')
    bse_art = fetch_google_news(
        f'{company_name} BSE announcement results board meeting', max_results=3
    )
    print_articles(bse_art, indent=4)
    for a in bse_art:
        all_news.append({'Section': 'Company', 'Category': company_name, 'Query': 'BSE Announcement', **a})
    time.sleep(0.4)

    print('  yfinance headlines:')
    yf_art = fetch_yfinance_news(ticker, max_results=3)
    print_articles(yf_art, indent=4)
    for a in yf_art:
        all_news.append({'Section': 'Company', 'Category': company_name, 'Query': 'yfinance', **a})
    time.sleep(0.4)

  SECTION 3: COMPANY NEWS

───────────────────────────────────────────────────────
  Maruti Suzuki (MARUTI.NS)
  ─────────────────────────────────────────────────────
  Google News:
    1. [2026-05-22 06:40] Autocar India
       Maruti Suzuki to hike prices by up to Rs 30,000 in June - Autocar India
       https://news.google.com/rss/articles/CBMiowFBVV95cUxNbU1CM2huQ2c1ZDJDdFdKM2dnUTNGMk9qZ3ZJRFBINjZ0dXBQZnFHNHRtcDU4T3BHRVpMbl93N3h4QWdhZmg0d3lCUW11RkVYRjFyZ1JIdlJwdURHWkM0RTVwZWxSakQ4bUc2dVhwUThSVjVOUG5pakVfWElZZEthb0o0Qjktenc4T0V4eHZZV3FvTlduOFlMRHpTS0lUZlZXLW1F0gGoAUFVX3lxTE9tTkpiM1pwYnI3Ql9saWN2QWthNk85bk9aTE1vM2Z0U0pjcFItVHNUdkRpdmo4Ui1mN1R3VjBRbEM2RVBMTTgwS29kQnMzdThwXzJRazNWZ2xkOEtNVk5RQkc2MmRXdnF1b0FDc25wdy1kaFc2REJsOGhXNWtzSUlWUGFjR21Vcy1GMmN5UUdIc3h6Y0t4QTdScXM3T0NsY2FxS0dUcDZCSQ?oc=5

    2. [2026-05-23 15:43] Business Today
       Maruti Suzuki to launch flex-fuel car that runs on 100% ethanol on June 5: Nitin Gadkari - Business Today
       https://news.google.com/rss/artic

In [8]:
# ── SECTION 4: Your Notes ─────────────────────────────────────────────────────
# After reading sections 1-3, fill in your assessments here.
# These get saved alongside the headlines in the Excel export.

YOUR_NOTES = {
    'macro_assessment' : '',   # e.g. 'RBI held rates, neutral. No budget surprise.'
    'sector_assessment': '',   # e.g. 'FMCG demand strong pre-summer. No policy risk.'
    'stock_notes'      : {
        # 'HINDUNILVR.NS': 'Q4 results Apr 28 — historically beats. Hold through.',
        # 'ITC.NS'       : 'Cigarette tax rumor — watch closely.',
    },
    'overall_go_no_go' : '',   # 'GO' or 'NO GO' or 'PARTIAL (exclude X)'
    'notes_date'       : datetime.now().strftime('%Y-%m-%d'),
}

print('Fill in YOUR_NOTES above after reading the sections, then run the export cell below.')

Fill in YOUR_NOTES above after reading the sections, then run the export cell below.


In [ ]:
# ── Export to Excel ───────────────────────────────────────────────────────────

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

timestamp   = datetime.now().strftime('%Y%m%d_%H%M')
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f'news_{ACTIVE_WINDOW.replace(" ","_")}_{timestamp}.xlsx')

# Build DataFrames
news_df = pd.DataFrame(all_news).drop_duplicates(subset='title').reset_index(drop=True)
news_df = news_df.sort_values(['Section', 'Category', 'published'], ascending=[True, True, False])

notes_rows = [
    {'Category': 'Macro',           'Note': YOUR_NOTES['macro_assessment']},
    {'Category': 'Sector',          'Note': YOUR_NOTES['sector_assessment']},
    {'Category': 'Overall Go/No-Go','Note': YOUR_NOTES['overall_go_no_go']},
    {'Category': 'Date',            'Note': YOUR_NOTES['notes_date']},
]
for ticker, note in YOUR_NOTES['stock_notes'].items():
    notes_rows.append({'Category': ticker, 'Note': note})
notes_df = pd.DataFrame(notes_rows)

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    news_df[news_df['Section']=='Macro'  ].to_excel(writer, sheet_name='Macro News',    index=False)
    news_df[news_df['Section']=='Sector' ].to_excel(writer, sheet_name='Sector News',   index=False)
    news_df[news_df['Section']=='Company'].to_excel(writer, sheet_name='Company News',  index=False)
    news_df                               .to_excel(writer, sheet_name='All Headlines', index=False)
    notes_df                              .to_excel(writer, sheet_name='My Notes',      index=False)

# Formatting
wb = load_workbook(OUTPUT_PATH)

HEADER_FILL  = PatternFill('solid', fgColor='1F3864')
MACRO_FILL   = PatternFill('solid', fgColor='E8F4FD')
SECTOR_FILL  = PatternFill('solid', fgColor='EAF3DE')
COMPANY_FILL = PatternFill('solid', fgColor='FFF8E7')
ALT_FILL     = PatternFill('solid', fgColor='F5F5F5')
HEADER_FONT  = Font(bold=True, color='FFFFFF', size=10)
LINK_FONT    = Font(color='0563C1', underline='single', size=9)
THIN_BORDER  = Border(bottom=Side(style='thin', color='DDDDDD'))
SECTION_FILLS = {'Macro': MACRO_FILL, 'Sector': SECTOR_FILL, 'Company': COMPANY_FILL}

COL_WIDTHS = {
    'Section': 10, 'Category': 25, 'Query': 22,
    'title'  : 65, 'source'  : 20, 'published': 16, 'link': 45,
    'Note'   : 80,
}

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    headers  = [c.value for c in ws[1]]
    sec_col  = headers.index('Section') + 1 if 'Section' in headers else None
    link_col = headers.index('link')    + 1 if 'link'    in headers else None

    for cell in ws[1]:
        cell.fill = HEADER_FILL; cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border = THIN_BORDER
    ws.row_dimensions[1].height = 22
    ws.freeze_panes = 'A2'

    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        sec_val   = row[sec_col - 1].value if sec_col else None
        base_fill = SECTION_FILLS.get(sec_val, ALT_FILL if row_idx % 2 == 0 else PatternFill())
        for cell in row:
            cell.fill = base_fill; cell.border = THIN_BORDER
            cell.alignment = Alignment(horizontal='left', vertical='center')
        ws.row_dimensions[row_idx].height = 15
        if link_col and row[link_col - 1].value:
            row[link_col - 1].font = LINK_FONT

    for col in ws.columns:
        header = col[0].value
        ws.column_dimensions[get_column_letter(col[0].column)].width = COL_WIDTHS.get(header, 15)

wb.save(OUTPUT_PATH)
print(f'\n✅ News report saved: {OUTPUT_PATH}')
print(f'   Total headlines : {len(news_df)}')
print(f'   Macro           : {len(news_df[news_df["Section"]=="Macro"])}')
print(f'   Sector          : {len(news_df[news_df["Section"]=="Sector"])}')
print(f'   Company         : {len(news_df[news_df["Section"]=="Company"])}')
print(f'\nSheets: Macro News | Sector News | Company News | All Headlines | My Notes')